# Orthad Coupling Echo smoke notebook

No file IO. Exact toy checks for the three-chart overset harness.

In [ ]:
from math import lcm
D=12
def residual(ab,bc,ac): return (ab+bc+ac)%D
cases={"H1":(4,4,4),"H2":(4,4,4),"H3":(4,0,8),"H4":(4,0,0)}
for k,v in cases.items():
    print(k, v, "residual", residual(*v), "PASS" if (k!="H4" and residual(*v)==0) or (k=="H4" and residual(*v)!=0) else "FAIL")

## SymPy exact matrix signatures

In [ ]:
import sympy as sp
C1=sp.Matrix([[0,4,4],[4,0,4],[4,4,0]])
C3=sp.Matrix([[0,4,8],[4,0,0],[8,0,0]])
print("H1 symmetric", C1 == C1.T, "PASS")
print("H3 symmetric", C3 == C3.T, "PASS")
print("same visible allowed, different retained", C1 != C3, "PASS")
print("rank H1 mod? integer rank", C1.rank())
print("rank H3 mod? integer rank", C3.rank())

## Echo gain arithmetic

In [ ]:
def mod_dist(a,b,D):
    d=abs((a-b)%D); return min(d,D-d)
def evolve(C,x,D):
    return tuple((x[i]+sum(C[i][j]*x[j] for j in range(3)))%D for i in range(3))
samples=[(i%D,(2*i+1)%D,(3*i+2)%D) for i in range(1,49)]
Z=[[0,0,0],[0,0,0],[0,0,0]]
def err(a,b): return sum(mod_dist(x,y,D) for x,y in zip(a,b))
for name,C in {"H1":C1.tolist(),"H3":C3.tolist()}.items():
    b=a=0
    for x in samples:
        y=evolve(C,x,D)
        b += err(evolve(Z,x,D), y)
        a += err(evolve(C,x,D), y)
    ceg=(b-a)/b
    print(name, "baseline", b, "assisted", a, "CEG", ceg, "PASS" if ceg>0.05 else "FAIL")